# Ch.3  機械学習で品種を分類する

【講師用完全版】

| 項目 | 内容 |
|------|------|
| 使うデータ | ワインデータ 178件・13特徴量・3品種 |
| 演習時間 | 35分（問3まで完了で合格） |
| ゴール | RandomForest でモデルを作り、正解率と特徴量重要度を確認する |

---
### 📌 講師メモ：時間配分と時間が押した場合の対応

| 区分 | 内容 | 目安時間 |
|------|------|---------|
| 座学前半（昼前） | 教師あり学習・X/y・データ分割・過学習 | 15分 |
| 座学後半（昼後） | 決定木・RF・混同行列・評価指標 | 15分 |
| 演習（問1〜問2） | モデル学習・正解率 | 20分 |
| 演習（問3） | 混同行列・特徴量重要度（最重要） | 15分 |
| 問4（発展） | Precision / Recall | 余り次第 |

**時間が押した場合：**
- 問3の特徴量重要度（Q3-2）は Ch.2 の仮説との答え合わせとして必ず実施する
- 問4（Precision/Recall）は Ch.5 発展で扱うため、スキップ可
- **正解率と混同行列まで確認できれば Ch.4 に進んで問題ない**

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇をしたい / 〇〇を確認したい
【使うライブラリ】scikit-learn / pandas
【データの形】df という DataFrame、178行×14列（数値13列 + 品種名列）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

💡 Ch.3 の一言アドバイス：
「使うクラス名（RandomForestClassifier など）と、X・y の形を具体的に伝えると
 精度の高いコードが返ってきます。」

「モデルを作りたい」ではなく、
「X_train と y_train を使って RandomForestClassifier を学習させたい、どう書くか」のように、
**変数名・クラス名・やりたいこと** をセットで伝えましょう。

---
## 準備  ライブラリとデータを読み込む

⛔ 注意 最初にこのセルを実行してください。

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import japanize_matplotlib

wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種名"] = ["wine_" + str(t) for t in wine.target]
print(f"行数: {len(df)}  列数: {len(df.columns)}")

---
## なぜ「機械学習」を使うのか

Ch.1・Ch.2 で「alcohol と proline で品種が分かれそう」という仮説を立てました。

人間が「alcohol >= 13.5 かつ proline >= 700 なら wine_0」と手動でルールを作ることも可能ですが：

| 方法 | 問題点 |
|------|--------|
| 人間のルール | 変数が 13 個あると組み合わせが膨大 |
| 人間のルール | 新しいデータに対応できない |
| 機械学習（RF） | 13 変数すべてを使って最適なルールを自動生成 |

今回使う RandomForest：
- 決定木（木の形のルール）をたくさん作って多数決する
- 「どの変数がどれだけ判断に使われたか（特徴量重要度）」がわかる

---
## STEP 1  モデルへの「入力」と「答え」を分ける

機械学習の学習に必要なのは：
- X（特徴量）: モデルが「見る」数値データ → 13 列の数値
- y（ターゲット）: モデルが「当てる」答え → 品種（0/1/2）

---

Q1-1：特徴量（X）とターゲット（y）に分けてみましょう

- X：品種名を除いた数値列 13 個
- y：品種名列

💡 ヒント Copilot への相談例：
「pandas で特定の列を除いた DataFrame と、その列だけを別変数に分けたい」

In [ ]:
# 品種名を除いた数値列を X（特徴量）、品種名列を y（ターゲット）として分ける
X = df.drop("品種名", axis=1)   # 13 列の数値特徴量
y = df["品種名"]                 # 予測する品種ラベル
print(f"X: {X.shape}  y: {y.shape}")

---
### 📊 出力結果の解説

```
X: (178, 13)  y: (178,)
```

| 変数 | 意味 | 形 |
|------|------|---|
| X | モデルが「見る」13 特徴量 | 178行 × 13列 |
| y | モデルが「当てる」品種ラベル | 178行 × 1列 |

📌 ポイント X と y はセットで扱う。行数が同じ（178）であることが必須。

📌 講師メモ: 「X と y の行数が同じかを確認する習慣を付けましょう」と声がけ。

---
Q1-2：データを「学習用」と「テスト用」に分けましょう

モデルは学習に使ったデータでは高い精度が出ます。
未知のデータ（テスト用）でも正しく予測できるかを確認するため、事前に分割します。

学習用：テスト用 = 8：2 に分割してください。

💡 ヒント Copilot への相談例：
「scikit-learn で DataFrame を学習用とテスト用に 80:20 に分割したい」

In [ ]:
# 学習用（80%）とテスト用（20%）にランダムに分割する（再現性のため random_state を固定）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"学習: {len(X_train)}件  テスト: {len(X_test)}件")

---
### 📊 出力結果の解説

```
学習: 142件  テスト: 36件
```

📌 ポイント random_state=42 を指定すると、毎回同じ分け方になる（再現性の確保）。
テスト用は「モデルが見たことがないデータ」として保管する。
テスト用で高い精度が出れば「汎化性能が高いモデル」と評価できる。

📌 講師メモ: 「なぜデータを分ける必要があるのか？」を問い返し、過学習の概念を簡単に説明。

---
## 問1  モデルを学習させる ★☆☆

実務でのイメージ：
「過去の顧客データ（X_train, y_train）をもとに、新規顧客（X_test）の行動を予測するモデルを作る」

---

Q：RandomForestClassifier でモデルを作り、学習させてください

1. `model = RandomForestClassifier(...)` でモデルを作る
   📌 `(...)` には `n_estimators=100, random_state=42` を入れます。n_estimators はランダムに作る決定木の本数です（多いほど安定）。
2. `model.fit(X_train, y_train)` で学習させる

💡 ヒント Copilot への相談例：
「scikit-learn の RandomForestClassifier で分類モデルを作って、学習データで学習させたい」

In [ ]:
# RandomForest モデルを作成して、学習データで学習させる
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("学習完了")

---
### 📊 出力結果の解説

```
学習完了
```

📌 ポイント `n_estimators=100` は「決定木を 100 本作って多数決する」という設定。
多くすると安定するが計算時間が増える。少なすぎると不安定。100 は標準的な設定。
`random_state=42` で再現性を確保。

📌 講師メモ: 「RandomForest が何をしているか」を 1 分で補足：
「100 本の木それぞれが予測を出して、多数決で最終結果を決める仕組みです」と説明。

✅ （模範解答）`model.fit(X_train, y_train)` でエラーなく「学習完了」と表示されれば成功。

---
## 問2  テストデータで予測して正解率を確認する ★★☆

実務でのイメージ：
「新しい顧客データに対してモデルが正しく品種を当てられるか、正解率で評価する」

---

Q：テストデータで予測して、正解率を計算してください

1. `model.predict(X_test)` で予測する
2. `accuracy_score(y_test, y_pred)` で正解率を計算する

💡 ヒント Copilot への相談例：
「scikit-learn で学習済みモデルでテストデータを予測して、正解率を計算したい」

In [ ]:
# テストデータで予測して、正解率（accuracy）を計算する
y_pred = model.predict(X_test)
acc   = accuracy_score(y_test, y_pred)
print(f"正解率: {acc:.3f}  ({acc*100:.1f}%)")

---
### 📊 出力結果の解説

```
正解率: 0.972  (97.2%)
```

| 確認するポイント | 見るべきこと |
|----------------|------------|
| 正解率 ≈ 97% | 36件中 35件正解（1件だけ間違い） |
| 高い理由 | Ch.1・Ch.2 で見たように、品種間の差が明確なデータだったから |

📌 ポイント 97% は非常に高い正解率。実務では 70〜80% でも「使えるモデル」の場合がある。
正解率だけでなく「どこで間違えたか」も重要 → 次の問3（混同行列）へ。

📌 講師メモ: 「もし正解率が 60% だったらどう思う？」と問い返し、目安の感覚を持たせる。

✅ （模範解答）正解率 97% 程度が期待される。「どの品種を間違えたか」は混同行列で確認する。

---
### 問2 の気づき（AI 禁止：1 行でOK）

Q：正解率は何 % でしたか？目標の 90% を超えていましたか？

>

✅ （模範解答）97% 程度（36件中 35件正解）。目標の 90% を大きく超えている。wine データは品種間の差が明確なため、RandomForest で高精度が出やすい。

📌 講師メモ：「もし 60% だったらどう思う？」と問い返し、正解率の"感覚"を持たせる。

---
## 問3  「どこで間違えたか」と「何が重要か」を確認する ★★☆（最重要）

実務でのイメージ：
「どの品種（顧客セグメント）を間違えやすいか」を特定して改善する

---

Q3-1：混同行列を表示して、どの品種を間違えたか確認してください

混同行列は「実際の品種」×「予測した品種」の表です。
対角線以外のセルに数値があると「間違い」です。

💡 ヒント Copilot への相談例：
「scikit-learn の confusion_matrix で混同行列を計算して、pandas の DataFrame で見やすく表示したい」

In [ ]:
# 混同行列を計算して、どの品種を間違えたかを確認する
cm     = confusion_matrix(y_test, y_pred)
classes = model.classes_
cm_df   = pd.DataFrame(cm, index=classes, columns=classes)
print("混同行列（行=実際、列=予測）")
print(cm_df)

---
### 📊 出力結果の解説

```
混同行列（行=実際、列=予測）
        wine_0  wine_1  wine_2
wine_0      14       0       0
wine_1       0      14       0
wine_2       0       1       7
```

| 読み方 | 内容 |
|--------|------|
| 行（横）| 実際の品種 |
| 列（縦）| モデルが予測した品種 |
| 対角線の数値 | 正解件数（大きいほどよい） |
| 対角線以外 | 間違い件数（ゼロが理想） |

この例では wine_2 を 1 件だけ wine_1 と間違えた。

📌 ポイント どの品種がどの品種に間違われやすいかが見える = 改善の方向性が見える。
「wine_2 の予測精度を上げるには、wine_2 だけのデータを増やす or 特徴量を追加する」と考えられる。

📌 講師メモ: 「wine_1 と wine_2 はどの変数が似ていましたか？（Ch.2 を振り返る）」と発問。

✅ （模範解答）対角線の値が最大になっていれば正しい。wine_2 が 1件 wine_1 と誤分類されることが多い。

---
### Q3-1 の気づき（AI 禁止：1 行でOK）

Q：対角線以外のセルに数値はありましたか？あった場合、どの品種を間違えていましたか？

>

✅ （模範解答）wine_2 が 1件 wine_1 と誤分類された。wine_2 と wine_1 はアルコール度数が近いため。Ch.2 の箱ひげ図で重なりがあると気づいたペアと一致する可能性がある。

📌 講師メモ：「Ch.2 でどの品種が見分けにくいと書きましたか？」と Ch.2 の記録を振り返らせる。

---
Q3-2：特徴量重要度を棒グラフで表示してください

RandomForest は「どの変数が品種の分類に役立ったか（重要度）」を自動的に計算します。
Ch.2 で立てた仮説（alcohol・proline が重要そう）は当たっていましたか？

💡 ヒント Copilot への相談例：
「scikit-learn の RandomForest で特徴量重要度を取得して、棒グラフで表示したい。重要度が高い順に並べたい」

In [ ]:
# 特徴量重要度を取得して、重要度が高い順に横棒グラフで表示する
importances = model.feature_importances_
feat_df     = pd.DataFrame({"特徴量": X.columns, "重要度": importances})
feat_df     = feat_df.sort_values("重要度", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(feat_df["特徴量"], feat_df["重要度"])
plt.title("特徴量重要度（RandomForest）")
plt.xlabel("重要度")
plt.tight_layout()
plt.show()

---
### 📊 出力結果の解説

上位 3 変数（目安）：
1. proline（最重要）
2. color_intensity
3. flavanoids

| Ch.2 での仮説 | 結果 | 一致？ |
|-------------|------|--------|
| proline が重要そう | 1位 | 一致 |
| alcohol が重要そう | 4〜5位 | おおむね一致 |
| flavanoids が重要そう | 3位 | 一致 |

📌 ポイント グラフで立てた仮説が機械学習で「数値として証明された」 = EDA の価値。
重要度上位の変数を「なぜこの変数が重要か」とビジネス視点で解釈することが実務で大切。

📌 講師メモ: 「Ch.2 の仮説は当たっていましたか？」と問い返す。

✅ （模範解答）proline が最重要。Ch.2 で「ばらつきが最大で品種間の差が最大」と気づいた変数と一致する。
EDA（探索的データ分析）が機械学習の結果と一致することで、分析の整合性が確認できる。

---
### Q3-2 の気づき（AI 禁止：1 行でOK）

Q：特徴量重要度 1 位の変数は何でしたか？Ch.2 の仮説は当たっていましたか？

>

✅ （模範解答）proline が 1位。Ch.2 で「std（標準偏差）が最大で品種間の差が最大」と気づいた変数と一致する。Ch.1 → Ch.2 → Ch.3 の分析の流れが整合している。

📌 講師メモ：「グラフで立てた仮説が数値で証明された」ことを強調し、EDA の価値を印象付ける。

---
## STEP 3  考察・振り返り （AI 禁止）

⛔ 注意 AI は使わないこと。自分の言葉で書いてください。

Q：Ch.2 で立てた仮説（重要な変数の予想）は当たっていましたか？

>

Q：正解率が約 97% というのは「良いモデル」だと思いますか？その理由は？

💡 参考：実務では 70〜80% でも「使えるモデル」として採用されることがあります。何 % 以上が合格かはビジネスの文脈によります。

>

---
### 📌 講師メモ（考察の模範解答）

（模範解答 1）仮説の検証：
proline・flavanoids は Ch.2 の仮説通り。
alcohol は予想より低い順位だった場合 → 「alcohol だけでは品種を分けにくかったが、他の変数と組み合わせると有効」と解説。

（模範解答 2）97% の評価：
良いモデルと言える。ただし今回はデータが 178件と少なく、品種間の差も明確だったため高く出やすい。
実務では 70〜80% でも十分に価値があるケースも多い。「何%以上が合格か」はビジネスの文脈によって異なる。

よくある受講生の詰まり方：
- 「97% は低い？」→ 「何件中何件正解か計算してみよう（36件中 35件正解）」と具体化する

---
## 問4（発展）  予測の詳細を確認する ★★★

📋 発展 時間が余った人だけ取り組んでください

実務でのイメージ：

📌 **Precision（適合率）**：「wine_X と予測した中で、本当に wine_X だった割合」
　例）「陽性と診断した患者のうち、本当に病気だった割合」

📌 **Recall（再現率）**：「本当に wine_X のうち、正しく wine_X と予測できた割合」
　例）「本当に病気の患者のうち、正しく陽性と診断できた割合」

→ 「見逃したくない（病気なのに陰性と言いたくない）」→ **Recall を高める**
→ 「誤診をなくしたい（病気でないのに陽性と言いたくない）」→ **Precision を高める**

---

Q：品種ごとの Precision・Recall を計算して表示してください

💡 ヒント Copilot への相談例：
「scikit-learn で分類モデルの品種ごとの Precision と Recall を計算したい（average='macro'）」

In [ ]:
# 品種ごとの Precision・Recall を計算して確認する
from sklearn.metrics import precision_score, recall_score
prec   = precision_score(y_test, y_pred, average=None)
rec    = recall_score(y_test, y_pred, average=None)
pr_df  = pd.DataFrame({"品種": model.classes_, "Precision": prec.round(3), "Recall": rec.round(3)})
print(pr_df)

---
### 📊 出力結果の解説

```
     品種  Precision  Recall
0  wine_0      1.000   1.000
1  wine_1      0.933   1.000
2  wine_2      1.000   0.875
```

| 指標 | 意味 | この結果 |
|------|------|---------|
| Precision | 「wine_X と予測した中で、本当に wine_X だった割合」 | wine_1 が 0.93 → 予測の 7% は誤り |
| Recall | 「本当に wine_X のうち、正しく wine_X と予測できた割合」 | wine_2 が 0.875 → 1件見逃し |

✅ （模範解答）wine_2 の Recall が 0.875 = 実際の wine_2 のうち 12.5%（1件）を見逃している。
wine_1 の Precision が 0.933 = wine_1 と予測した中の 6.7%（1件）は実際は別品種だった。

---
## お疲れさまでした！

| 操作 | 発見 | ビジネスへの接続 |
|------|------|----------------|
| モデル学習 | 13 変数で自動的にルールを作成 | 人間のルール作りを自動化 |
| 正解率 97% | ほぼすべての品種を正しく分類 | 現場で使えるモデルの基準 |
| 混同行列 | wine_2 の一部が wine_1 に誤分類 | 改善が必要な品種がわかる |
| 特徴量重要度 | proline が最重要 → Ch.2 の仮説と一致 | EDA → ML の流れが完結 |